In [1]:
from pathlib import Path

# Get project root
try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"
data_processed.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT:", PROJECT_ROOT)

PROJECT ROOT: /workspaces/flood-ml-research


In [2]:
import pandas as pd

base_path = data_processed / "makassar_spatiotemporal_base.parquet"

df = pd.read_parquet(base_path)

print("DATA LOADED:", df.shape)
df.head()

DATA LOADED: (5174884, 2)


,grid_id,date
0,227,2018-01-01
1,227,2018-01-02
2,227,2018-01-03
3,227,2018-01-04
4,227,2018-01-05


In [3]:
import numpy as np

np.random.seed(42)

# Rainfall (mm/day)
df["rainfall"] = np.random.gamma(
    shape=2.0,
    scale=10.0,
    size=len(df)
)

print(df["rainfall"].describe())

count    5.174884e+06
mean     1.999862e+01
std      1.413415e+01
min      4.311550e-03
25%      9.613088e+00
50%      1.678547e+01
75%      2.692445e+01
max      1.977217e+02
Name: rainfall, dtype: float64


In [4]:
threshold = df["rainfall"].quantile(0.95)

df["extreme_rain"] = (df["rainfall"] > threshold).astype(int)

print("Threshold:", threshold)
print("Extreme ratio:", df["extreme_rain"].mean())

Threshold: 47.43538584893308
Extreme ratio: 0.05000015459283725


In [5]:
# Proxy flood label
df["flood_label"] = df["extreme_rain"]

print("Flood ratio:", df["flood_label"].mean())

Flood ratio: 0.05000015459283725


In [6]:
df["month"] = df["date"].dt.month

monthly_flood = df.groupby("month")["flood_label"].mean()

print(monthly_flood)

month
1     0.051032
2     0.049265
3     0.049821
4     0.050409
5     0.049377
6     0.050205
7     0.050224
8     0.049942
9     0.050089
10    0.049926
11    0.049706
12    0.049951
Name: flood_label, dtype: float64


In [ ]:
df.head()

In [8]:
parquet_out = data_processed / "makassar_labeled.parquet"

df.to_parquet(parquet_out, index=False)

print(f"✔ PARQUET SAVED -> {parquet_out}")

✔ PARQUET SAVED -> /workspaces/flood-ml-research/data/processed/makassar_labeled.parquet


In [9]:
csv_out = data_processed / "makassar_labeled.csv"

df.to_csv(csv_out, index=False)

print(f"✔ CSV SAVED -> {csv_out}")

✔ CSV SAVED -> /workspaces/flood-ml-research/data/processed/makassar_labeled.csv
